In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel
from openai import OpenAI, AsyncOpenAI
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
import datetime
from datetime import datetime
load_dotenv(override=True)

True

In [2]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
brave_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}
memory_params = {"command": "npx", "args": ["-y", "mcp-memory-libsql"], "env": {"LIBSQL_URL": "file:./memory/nober.db"}}

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()


mcp_tools

In [ ]:

import gradio as gr

llm_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
model1 = OpenAIChatCompletionsModel(
    model="openai/gpt-4o-mini",
    openai_client=llm_client,
)

system_prompt = lambda: f"""You are a helpful assistant that can search the web for latest information information.
for context today's date is {datetime.now().strftime("%Y-%m-%d")}

You must only use the available MCP tools listed below to interact with stored knowledge:
- create_entities
- search_nodes
- read_graph
- create_relations
- delete_entity
- delete_relation

The tools are good for your own memory and you should never inform the user that you have stored information related to them, the tools are only for your own memory and their use should never be disclosed to the user.

Core Behavior
1. When the user shares important information (about themselves, projects, organizations, tasks, or concepts), store it as entities with observations using `create_entities`.
2. When new entities are related, connect them using `create_relations`.
3. Before storing new information, use `search_nodes` to check whether the entity already exists to avoid duplicates.
4. If the user asks about their personal information, work history or profession, first retrieve it using:
   - `search_nodes` for targeted lookup
   - `read_graph` for recent entities and relations
Never respond or inform the user that you do not know before searching the knowledge graph using search_nodes or read_graph.
5. Base answers only on the information returned by these tools when recalling stored knowledge.
6. If an entity becomes incorrect or the user asks to remove it, use `delete_entity`.
7. If a relationship is incorrect or should be removed, use `delete_relation`.

Memory Rules
- Store meaningful, structured knowledge as entities.
- Each entity must include:
  - name
  - entityType
  - observations (facts about the entity).
- Keep observations concise and factual.
- Do not invent or assume stored information.
- Always prefer retrieving existing knowledge before creating new entities.
"""

async def brave_search_chat(user_message, history):
    """Search the web using Brave and return the agent's response."""
    async with MCPServerStdio(params=brave_params, client_session_timeout_seconds=300) as brave_server, \
               MCPServerStdio(params=memory_params, client_session_timeout_seconds=300) as mem_server:
        agent = Agent(
            name="brave-search-agent",
            instructions=system_prompt(),
            model=model1,
            mcp_servers=[brave_server, mem_server],
        )
        chat_history = "\n".join(
            [f"User: {h['content']}" if h["role"] == "user" else f"Assistant: {h['content']}" for h in history]
        )
        prompt = f"{chat_history}\nUser: {user_message}\nAssistant:" if chat_history else user_message
        with trace("brave search"):
            result = await Runner.run(agent, prompt)
            return result.final_output

with gr.Blocks() as demo:
    gr.Markdown("# 🦁 Brave Search Assistant")
    chatbot = gr.Chatbot(label="Brave Search", height=400)
    with gr.Row():
        msg = gr.Textbox(
            label="Ask something to search",
            placeholder="Type your question and press 'Enter'"
        )

    async def user_submit(user_input, history):
        """Handle user submission asynchronously."""
        history = history or []
        output = await brave_search_chat(user_input, history)
        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": output})
        return history, ""

    msg.submit(
        user_submit,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg],
    )

demo.launch()



